In [8]:
#!/usr/bin/env Rscript

suppressPackageStartupMessages({
  libs <- c("optparse","data.table","NbClust","ggplot2")
  for (p in libs) if (!requireNamespace(p, quietly = TRUE)) install.packages(p)
  lapply(libs, library, character.only = TRUE)
})

#========================
# ユーティリティ
#========================
read_numeric_matrix <- function(path){
  df <- read.csv(path, header = TRUE, row.names = 1, check.names = FALSE, stringsAsFactors = FALSE)
  is_num <- vapply(df, is.numeric, logical(1))
  as.matrix(df[, is_num, drop = FALSE])
}

dir_create <- function(...) {
  d <- file.path(...)
  if (!dir.exists(d)) dir.create(d, recursive = TRUE, showWarnings = FALSE)
  d
}

timestamp <- function() format(Sys.time(), "%Y%m%d_%H%M%S")

msg <- function(...) cat(sprintf(...), "\n")

#========================
# MDS & 次元選択セーフ化
#========================
compute_mds_from_corr <- function(numData, k_max = 50){
  NF <- ncol(numData)
  if (is.null(NF) || NF < 2) stop("入力の数値列が2列未満です")
  corData <- suppressWarnings(cor(numData, use = "pairwise.complete.obs"))
  corData[is.na(corData)] <- 0
  ddata <- dist(1 - abs(corData))
  k <- min(k_max, max(1, ncol(corData) - 1))
  mds <- cmdscale(ddata, k = k, eig = TRUE)
  eig_vals <- as.numeric(mds$eig)
  totaleig <- sum(eig_vals)
  peigen <- if (totaleig == 0) rep(0, length(eig_vals)) else eig_vals/totaleig
  contrib <- data.frame(
    axis = seq_along(eig_vals),
    eigenvalue = eig_vals,
    percent = peigen,
    cum_percent = cumsum(peigen),
    stringsAsFactors = FALSE
  )
  list(mds = mds, contrib = contrib)
}

compute_pos_contrib <- function(eig_vals) {
  totaleig <- sum(eig_vals)
  if (!is.finite(totaleig) || totaleig == 0) {
    return(list(percent = rep(0, length(eig_vals)), cum_percent = rep(0, length(eig_vals)), pos_idx = integer(0)))
  }
  peigen <- eig_vals / totaleig
  list(percent = peigen, cum_percent = cumsum(peigen), pos_idx = which(peigen > 0))
}

pick_k_top3   <- function(pos_idx)                    if (length(pos_idx)==0) 0L else as.integer(min(3L, length(pos_idx)))
pick_k_cumeig <- function(cump, pos_idx, thr = 0.80)  { if (length(pos_idx)==0) return(0L); k <- which(cump[pos_idx] >= thr)[1]; if (is.na(k)) k <- length(pos_idx); as.integer(k) }

slice_dims_safely <- function(coords_all, k_wanted) {
  p <- ncol(coords_all)
  if (is.null(p) || p < 1L) stop("MDS 座標の列数が不正です")
  k_eff <- as.integer(max(0L, min(k_wanted, p)))
  if (k_eff == 0L) return(NULL)
  coords_all[, seq_len(k_eff), drop = FALSE]
}

#========================
# NbClust 実行（Ward.D2）
#========================
run_nbclust_one <- function(X, index_name, kmin=2, kmax=25) {
  set.seed(.GlobalEnv$.SEED) # 再現性
  res <- tryCatch({
    NbClust(data = X, diss = NULL, distance = "euclidean",
            min.nc = kmin, max.nc = min(kmax, nrow(X)-1),
            method = "ward.D2", index = index_name)
  }, error = function(e) e)
  res
}

extract_best_k <- function(nb) {
  k_val <- NA_integer_
  if (inherits(nb, "error")) return(k_val)
  if (!is.null(nb$Best.nc)) {
    if (is.list(nb$Best.nc) && !is.null(nb$Best.nc[["Number_clusters"]])) {
      k_val <- as.integer(nb$Best.nc[["Number_clusters"]])
    } else if (is.matrix(nb$Best.nc) && "Number_clusters" %in% rownames(nb$Best.nc)) {
      k_val <- as.integer(nb$Best.nc["Number_clusters", 1])
    } else if (is.numeric(nb$Best.nc)) {
      k_val <- as.integer(nb$Best.nc[1])
    }
  }
  k_val
}

#========================
#  図出力（最小限）
#========================
save_scree <- function(contrib, out_png){
  ggplot(contrib, aes(x=axis, y=percent)) +
    geom_col() + geom_line(aes(y=cum_percent)) + geom_point(aes(y=cum_percent)) +
    labs(x="Axis", y="Proportion / Cumulative", title="MDS Scree / Cumulative") +
    theme_minimal()
  ggsave(out_png, width=7, height=4, dpi=150)
}

#========================
# 実行本体（1データセット）
#========================
process_dataset <- function(dataset, base_dir, out_root, kmin, kmax) {
  # 入力ファイル（決め打ち）
  in_map <- list(
#     "OH"=file.path(base_dir, "DataMerge20211220oh_20250717_OH_delSMILES.csv"),
#     "FP"=file.path(base_dir, "DataMerge20211220FP_20250715.csv")
    "OH"=file.path(base_dir, "preprocessed_features_OH.csv"),
    "FP"=file.path(base_dir, "preprocessed_features_FP.csv")

  )
      
  in_path <- in_map[[dataset]]
  if (is.null(in_path) || !file.exists(in_path)) stop(sprintf("[%s] 入力ファイルが見つかりません: %s", dataset, in_path))

  # 出力フォルダ
  out_ds   <- dir_create(out_root, dataset)
  d_eigen  <- dir_create(out_ds, "01_eigen")
  d_coords <- dir_create(out_ds, "02_coords")
  d_nb     <- dir_create(out_ds, "03_nbclust")
  d_assign <- dir_create(out_ds, "04_assignments")
  d_fig    <- dir_create(out_ds, "05_figures")
  d_logs   <- dir_create(out_ds, "99_logs")

  ts_tag <- timestamp()
  log_path <- file.path(d_logs, sprintf("STEP3.3_U25_runlog_%s_%s.txt", dataset, ts_tag))
  con <- file(log_path, open = "wt"); sink(con, type = "output"); on.exit({sink(); close(con)}, add=TRUE)

  msg("=== [%s] ===", dataset)
  msg("[INPUT] %s", in_path)
  num <- read_numeric_matrix(in_path)
  msg("[INFO] rows=%d variables=%d (numeric)", nrow(num), ncol(num))

  # MDS
  m <- compute_mds_from_corr(num, k_max = 50)
  mds <- m$mds
  contrib <- m$contrib
  write.csv(contrib, file.path(d_eigen, sprintf("MDS_eigen_contributions_%s_%s.csv", dataset, ts_tag)), row.names = FALSE)
  save_scree(contrib, file.path(d_eigen, sprintf("Fig_MDS_scree_%s_%s.png", dataset, ts_tag)))

  # 正の寄与軸から k を決定（top3 / cumeig）
  pc <- compute_pos_contrib(contrib$eigenvalue)
  k_top3_rel <- pick_k_top3(pc$pos_idx)
  k_cum_rel  <- pick_k_cumeig(pc$cum_percent, pc$pos_idx, thr = 0.80)
  coords_all <- as.matrix(mds$points)
  msg("[DEBUG] ncol(coords_all)=%d, pos_axes=%d, k_top3_rel=%d, k_cum_rel=%d",
      ncol(coords_all), length(pc$pos_idx), k_top3_rel, k_cum_rel)

  coords_top3 <- if (k_top3_rel>0) slice_dims_safely(coords_all, k_top3_rel) else NULL
  coords_cume <- if (k_cum_rel >0) slice_dims_safely(coords_all, k_cum_rel) else NULL

  if (!is.null(coords_top3)) {
    write.csv(coords_top3, file.path(d_coords, sprintf("MDScoords_top3_%s_%s.csv", dataset, ts_tag)))
  } else {
    warning("[top3] 正の寄与軸が0 → 出力スキップ")
  }
  if (!is.null(coords_cume)) {
    write.csv(coords_cume, file.path(d_coords, sprintf("MDScoords_cumeig_%s_%s.csv", dataset, ts_tag)))
  } else {
    warning("[cumeig] 正の寄与軸が0 → 出力スキップ")
  }

  # NbClust（index一括）
  indices <- c("silhouette","dunn","gap","ch","db","ptbiserial")
  modes <- list(
    "top3"  = coords_top3,
    "cumeig"= coords_cume
  )

  nb_sum <- data.table(mode=character(), index=character(), best_k=integer())
  for (md in names(modes)) {
    X <- modes[[md]]
    if (is.null(X)) { warning(sprintf("[%s] %s: X=NULL → スキップ", dataset, md)); next }
    for (idx in indices) {
      msg(".. NbClust [%s | %s] k=%d..%d", md, idx, kmin, kmax)
      nb <- run_nbclust_one(X, idx, kmin, kmax)
      if (inherits(nb, "error")) {
        warning(sprintf("NbClust failed: %s | %s | %s", md, idx, nb$message))
        next
      }
      k_best <- extract_best_k(nb)
      nb_sum <- rbind(nb_sum, data.table(mode=md, index=idx, best_k=k_best))
      # ベスト分割を保存（なければスキップ）
      if (!is.null(nb$Best.partition)) {
        lab <- as.integer(nb$Best.partition)
        out_assign <- file.path(d_assign, sprintf("ClusterAssign_%s_%s_%s_%s.csv", md, idx, dataset, ts_tag))
        fwrite(data.table(id = names(nb$Best.partition), cluster = lab), out_assign)
      }
      # 参考：サイズバー図（省メモリ・必要な場合のみ）
      if (!is.null(nb$Best.partition)) {
        lab <- as.integer(nb$Best.partition)
        dfb <- data.table(cluster = lab)[, .N, by = cluster]
        p <- ggplot(dfb, aes(x=factor(cluster), y=N)) +
          geom_col() + labs(x="Cluster", y="Size",
          title=sprintf("%s | %s | k=%s", md, idx, as.character(k_best))) +
          theme_minimal()
        ggsave(file.path(d_fig, sprintf("Fig_Cluster_sizes_%s_%s_%s_%s.png", md, idx, dataset, ts_tag)), p, width=6, height=4, dpi=150)
      }
    }
  }
  fwrite(nb_sum, file.path(d_nb, sprintf("nbclust_summary_%s_%s.csv", dataset, ts_tag)))
  msg("[DONE] %s", dataset)
}

#========================
# 引数処理（optparse安全版）
#========================
option_list <- list(
  optparse::make_option(c("-b","--base"), type="character", default=NULL, help="Base directory"),
  optparse::make_option(c("-d","--datasets"), type="character", default="OH,FP", help="Datasets: OH,FP"),
  optparse::make_option("--kmin", type="integer", default=2),
  optparse::make_option("--kmax", type="integer", default=25),
  optparse::make_option("--seed", type="integer", default=20251027),
  optparse::make_option("--date", type="character", default=format(Sys.Date(), "%Y%m%d"))
)
parser <- optparse::OptionParser(option_list = option_list)

safelite_parse_args <- function(parser) {
  # Jupyter等で余計な --args が混入しても落ちないよう防御
  raw <- commandArgs(trailingOnly = TRUE)
  ok <- tryCatch(optparse::parse_args(parser, args = raw), error=function(e) NULL)
  if (is.null(ok)) {
    warning("[WARN] optparse failed. Falling back to defaults.")
    return(list(
      base = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No",
      datasets = "OH,FP",
      kmin = 2, kmax = 25, seed = 20251027,
      date = format(Sys.Date(), "%Y%m%d")
    ))
  }
  ok
}

opt <- safelite_parse_args(parser)
if (is.null(opt$base)) {
  # 明示未指定なら既定
  opt$base <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
}
datasets <- strsplit(opt$datasets, ",\\s*")[[1]]
kmin <- as.integer(opt$kmin); kmax <- as.integer(opt$kmax)
.SEED <- as.integer(opt$seed)
date_tag <- opt$date

# ルート出力フォルダ
out_root <- dir_create(opt$base, sprintf("STEP3.3_U25_%s", date_tag))
msg("Output root: %s", out_root)

#========================
# 実行
#========================
for (ds in datasets) {
  tryCatch(
    process_dataset(ds, opt$base, out_root, kmin, kmax),
    error = function(e) { message(sprintf("[ERROR][%s] %s", ds, e$message)) }
  )
}
message("All done.")


[[1]]
 [1] "mclust"     "MASS"       "scales"     "ggrepel"    "optparse"  
 [6] "NbClust"    "ggplot2"    "data.table" "stats"      "graphics"  
[11] "grDevices"  "utils"      "datasets"   "methods"    "base"      

[[2]]
 [1] "mclust"     "MASS"       "scales"     "ggrepel"    "optparse"  
 [6] "NbClust"    "ggplot2"    "data.table" "stats"      "graphics"  
[11] "grDevices"  "utils"      "datasets"   "methods"    "base"      

[[3]]
 [1] "mclust"     "MASS"       "scales"     "ggrepel"    "optparse"  
 [6] "NbClust"    "ggplot2"    "data.table" "stats"      "graphics"  
[11] "grDevices"  "utils"      "datasets"   "methods"    "base"      

[[4]]
 [1] "mclust"     "MASS"       "scales"     "ggrepel"    "optparse"  
 [6] "NbClust"    "ggplot2"    "data.table" "stats"      "graphics"  
[11] "grDevices"  "utils"      "datasets"   "methods"    "base"

Warning message in safelite_parse_args(parser):
"[WARN] optparse failed. Falling back to defaults."


Output root: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.3_U25_20251027 


All done.



In [7]:
# #!/usr/bin/env Rscript

# # =========================
# # MDS → クラスタリング 一括実行（OH/FP 自動処理・旧3.3仕様）
# # - 相関→距離(1 - |cor|)→cmdscale（古典MDS）
# # - NbClust（ward.D2, Euclidean）× 指標6種（silhouette, dunn, gap, ch, db, ptbiserial）
# # - 次元条件: top3 / cumeig（正の固有値寄与率で累積 ≥ 80%）
# # - 出力: old3.3の慣習に合わせ、new_MDS_系ではなく旧来の out_dir 直下に保存
# # - 本改修は「入力を3.2に統一／前処理を3.2互換」に限定。他は旧3.3に準拠。
# # =========================

# suppressPackageStartupMessages({
#   pkgs <- c("NbClust","ggplot2","ggrepel","scales","MASS","mclust","data.table")
#   for (p in pkgs) if (!requireNamespace(p, quietly=TRUE)) install.packages(p)
#   invisible(lapply(pkgs, library, character.only=TRUE))
# })

# # --- 設定（入力パス）---
# # ★パスは環境に合わせて調整してください（旧3.3準拠）
# base_dir  <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
# dataset_files <- c(  # ← 3.2と同じファイル名に統一
#   "preprocessed_features_OH.csv",
#   "preprocessed_features_FP.csv"
# )

# # --- 共通設定（旧3.3踏襲） ---
# set.seed(42)  # 再現性
# index_list   <- c("silhouette", "dunn", "gap", "ch", "db", "ptbiserial")  # 指標6種
# ts_day       <- format(Sys.Date(), "%Y%m%d")                               # 日付タグ
# root_out     <- file.path(base_dir, paste0("olddata_MDS_", ts_day))        # ★旧3.3に合わせる
# dir.create(root_out, showWarnings = FALSE, recursive = TRUE)

# # --- ヘルパー（旧3.3の体裁を維持） ---
# pct <- function(x) sprintf("%.2f%%", 100 * x)

# # 3.2互換の前処理を行う読み込み（今回の改修点）
# # ・目的変数（PCEmax, Jsc, Voc, FF）を除去
# # ・英字を含む列（character-heavy）を除去
# # ・数値化（numeric列のみ返す）
# # ・除外列は返り値 attr に記録
# read_numeric_matrix_3_2style <- function(path){
#   df <- read.csv(path, header = TRUE, row.names = 1, check.names = FALSE, stringsAsFactors = FALSE)

#   removed <- data.table(column=character(), reason=character())

#   # 目的変数を除外
#   target_vars <- c("PCEmax","Jsc","Voc","FF")
#   drop_tgt <- intersect(colnames(df), target_vars)
#   if (length(drop_tgt)) {
#     removed <- rbind(removed, data.table(column=drop_tgt, reason="is_target"))
#     df <- df[, setdiff(colnames(df), target_vars), drop = FALSE]
#   }

#   # 「英字を含むセルがある列」を除外（3.2準拠のラフな文字列検出）
#   is_char <- vapply(df, function(col) {
#     any(grepl("[A-Za-z]", as.character(col)) & !is.na(col))
#   }, logical(1))
#   if (any(is_char)) {
#     removed <- rbind(removed, data.table(column=names(which(is_char)), reason="char/heavy_non_numeric"))
#     df <- df[, !is_char, drop = FALSE]
#   }

#   # 数値化（numeric列のみ返す：旧3.3の read_numeric_matrix と同等の戻りに合わせる）
#   is_num <- vapply(df, is.numeric, logical(1))
#   if (!any(is_num)) stop("数値列が見つかりません: ", path)
#   out <- as.matrix(df[, is_num, drop = FALSE])

#   attr(out, "removed_log") <- removed
#   return(out)
# }

# compute_mds <- function(numData, k_max = 300){
#   NS <- nrow(numData); NF <- ncol(numData)
#   message(sprintf("[Info] rows=%d, cols=%d", NS, NF))
#   nonNApct <- sum(!is.na(as.matrix(numData))) / (NS*NF) * 100
#   message(sprintf("[Info] Non-NA percentage: %.2f%%", nonNApct))

#   # |cor| → 1 - |cor|
#   corData <- suppressWarnings(cor(numData, use = "pairwise.complete.obs"))
#   corData[is.na(corData)] <- 0
#   ddata <- dist(1 - abs(corData))

#   mds_k <- min(k_max, max(1, NF - 1))
#   mdsdata <- cmdscale(ddata, k = mds_k, eig = TRUE)

#   # 固有値（全体）と、正の固有値のみでの寄与率（プロットやcumeig用）
#   eig_all <- mdsdata$eig
#   S_all   <- sum(eig_all)
#   share_all <- ifelse(S_all == 0, 0, eig_all / S_all)

#   eig_pos <- eig_all[eig_all > 0]
#   S_pos   <- sum(eig_pos)
#   share_pos_full <- rep(NA_real_, length(eig_all))
#   if (length(eig_pos) > 0 && S_pos > 0) {
#     share_pos_full[eig_all > 0] <- eig_all[eig_all > 0] / S_pos
#   }

#   cum_all <- cumsum(share_all)
#   tmp <- share_pos_full; tmp[is.na(tmp)] <- 0
#   cum_pos <- cumsum(tmp)

#   list(mdsdata = mdsdata,
#        eig_all = eig_all,
#        share_all = share_all,
#        share_pos_full = share_pos_full,
#        cum_all = cum_all,
#        cum_pos = cum_pos)
# }

# choose_dims <- function(share_pos_full, cum_pos, target = 0.80){
#   # 正の固有値寄与で top3 / cumeig の軸数を決める（旧3.3ロジックのまま）
#   pos_idx <- which(!is.na(share_pos_full) & share_pos_full > 0)
#   top3 <- min(3, max(1, length(pos_idx)))

#   if (length(pos_idx) == 0) {
#     return(list(top3 = 1, cumeig = 3))  # フォールバック（旧仕様維持）
#   }
#   k_cum <- which(cum_pos[pos_idx] >= target)[1]
#   if (is.na(k_cum)) k_cum <- length(pos_idx)
#   cumeig <- max(1, k_cum)

#   list(top3 = as.integer(top3), cumeig = as.integer(cumeig))
# }

# save_eigen_csv_and_plots <- function(out_dir, suffix_tag, ts_tag, eig_all){
#   # 正の固有値でスクリープロット（旧3.3の図仕様を維持）
#   eig_pos <- eig_all[eig_all > 0]
#   if (length(eig_pos) > 0) {
#     df_eig <- data.frame(
#       Dim  = seq_along(eig_pos),
#       Eig  = as.numeric(eig_pos)
#     )
#     df_eig$Prop <- df_eig$Eig / sum(df_eig$Eig)
#     df_eig$Cum  <- cumsum(df_eig$Prop)

#     # Scree
#     p_scree <- ggplot(df_eig, aes(x = Dim, y = Prop)) +
#       geom_col(width = 0.85, alpha = 0.9) +
#       geom_line(aes(y = Cum), linewidth = 1.1) +
#       geom_point(aes(y = Cum), size = 2) +
#       scale_y_continuous(labels = scales::percent_format(accuracy = 1),
#                          sec.axis = sec_axis(~ ., name = "Cumulative share")) +
#       labs(title = "Scree plot (positive eigenvalues only)",
#            subtitle = paste0("Classical MDS  |  ", suffix_tag),
#            x = "Dimension", y = "Variance share") +
#       theme_minimal(base_size = 12) +
#       theme(panel.grid.minor = element_blank(),
#             axis.title = element_text(face = "bold"),
#             plot.title = element_text(face = "bold"))
#     ggsave(file.path(out_dir, paste0("MDS_scree_", suffix_tag, "_", ts_tag, ".png")),
#            p_scree, width = 7, height = 5, dpi = 300)
#     ggsave(file.path(out_dir, paste0("MDS_scree_", suffix_tag, "_", ts_tag, ".pdf")),
#            p_scree, width = 7, height = 5, device = cairo_pdf)

#     # 固有値トップ50
#     topN <- min(50, nrow(df_eig))
#     df_eig50 <- df_eig[1:topN, , drop = FALSE]
#     p_bar50 <- ggplot(df_eig50, aes(x = Dim, y = Prop)) +
#       geom_col(width = 0.85, alpha = 0.95) +
#       labs(title = paste0("Top-", topN, " eigenvalue shares"),
#            subtitle = paste0("Classical MDS  |  ", suffix_tag),
#            x = "Dimension", y = "Variance share") +
#       scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
#       theme_minimal(base_size = 12) +
#       theme(panel.grid.minor = element_blank(),
#             axis.title = element_text(face = "bold"),
#             plot.title = element_text(face = "bold"))
#     ggsave(file.path(out_dir, paste0("MDS_bar_top", topN, "_", suffix_tag, "_", ts_tag, ".png")),
#            p_bar50, width = 7, height = 5, dpi = 300)
#     ggsave(file.path(out_dir, paste0("MDS_bar_top", topN, "_", suffix_tag, "_", ts_tag, ".pdf")),
#            p_bar50, width = 7, height = 5, device = cairo_pdf)
#   } else {
#     warning("正の固有値が無いため、スクリープロットとTop50図はスキップします。")
#   }
# }

# run_nbclust_one_index <- function(mds_points, cindex, out_dir, suffix_tag, ts_tag){
#   clustEst <- tryCatch({
#     NbClust(
#       data = mds_points, diss = NULL, distance = "euclidean",
#       min.nc = 2, max.nc = min(100, nrow(mds_points) - 1),
#       method = "ward.D2", index = cindex
#     )
#   }, error = function(e) { warning(paste("Index", cindex, "failed:", e$message)); NULL })
#   if (is.null(clustEst)) return(NULL)

#   best_nc <- clustEst$Best.nc[1]
#   grpname <- as.factor(clustEst$Best.partition)

#   # 割当CSV（旧3.3命名を踏襲）
#   fn_assign <- paste0(cindex, "_DataGrp_", suffix_tag, ".csv")
#   write.csv(grpname, file = file.path(out_dir, fn_assign))

#   # 1-2 軸散布図（旧3.3風）
#   df_plot <- data.frame(
#     MDS1    = mds_points[, 1],
#     MDS2    = mds_points[, 2],
#     Cluster = grpname,
#     ID      = rownames(mds_points)
#   )
#   rng <- range(c(df_plot$MDS1, df_plot$MDS2), na.rm = TRUE)
#   pad <- diff(rng) * 0.05; lims <- c(min(rng) - pad, max(rng) + pad)

#   p_scatter <- ggplot(df_plot, aes(x = MDS1, y = MDS2, color = Cluster)) +
#     stat_ellipse(aes(group = Cluster), type = "norm", level = 0.95,
#                  linewidth = 0.6, alpha = 0.25) +
#     geom_point(size = 2.2, alpha = 0.8) +
#     coord_equal(xlim = lims, ylim = lims, expand = TRUE) +
#     scale_color_hue(h = c(0, 360), l = 55, c = 90) +
#     labs(title = "MDS (Dim1 vs Dim2) — Ward.D2 × NbClust",
#          subtitle = paste0("Index: ", cindex, "  |  Best k = ", best_nc, "  |  ", suffix_tag),
#          x = "MDS Dimension 1", y = "MDS Dimension 2", color = "Cluster") +
#     theme_minimal(base_size = 12) +
#     theme(panel.grid.major = element_line(linewidth = 0.3, linetype = "dashed"),
#           panel.grid.minor = element_blank(),
#           axis.title = element_text(face = "bold"),
#           plot.title = element_text(face = "bold", size = 14, margin = margin(b = 4)),
#           plot.subtitle = element_text(size = 11, colour = "grey30"),
#           legend.position = "right",
#           legend.title = element_text(face = "bold"))
#   ggsave(file.path(out_dir, paste0("MDS12_scatter_", cindex, "_", suffix_tag, "_", ts_tag, ".png")),
#          p_scatter, width = 7, height = 6, dpi = 300)
#   ggsave(file.path(out_dir, paste0("MDS12_scatter_", cindex, "_", suffix_tag, "_", ts_tag, ".pdf")),
#          p_scatter, width = 7, height = 6, device = cairo_pdf)

#   # NbClust All.index 曲線（旧3.3の軽量版）
#   if (!is.null(clustEst$All.index)) {
#     idx_vals  <- as.numeric(clustEst$All.index)
#     k_labels  <- names(clustEst$All.index)
#     k_seq <- if (is.null(k_labels) || any(k_labels == "")) seq_along(idx_vals) else suppressWarnings(as.numeric(k_labels))
#     if (any(is.na(k_seq))) k_seq <- seq_along(idx_vals)

#     df_idx <- data.frame(k = k_seq, value = idx_vals)
#     p_idx <- ggplot(df_idx, aes(x = k, y = value)) +
#       geom_line(linewidth = 1) +
#       geom_point(size = 2) +
#       labs(title = paste0("NbClust All.index — ", cindex),
#            subtitle = paste0("Ward.D2 / Euclidean  |  ", suffix_tag),
#            x = "k", y = "Index value") +
#       theme_minimal(base_size = 12) +
#       theme(panel.grid.minor = element_blank(),
#             axis.title = element_text(face = "bold"),
#             plot.title = element_text(face = "bold"))
#     ggsave(file.path(out_dir, paste0("NbClust_AllIndex_", cindex, "_", suffix_tag, "_", ts_tag, ".png")),
#            p_idx, width = 7, height = 5, dpi = 300)
#     ggsave(file.path(out_dir, paste0("NbClust_AllIndex_", cindex, "_", suffix_tag, "_", ts_tag, ".pdf")),
#            p_idx, width = 7, height = 5, device = cairo_pdf)
#   }

#   # クラスタサイズ棒グラフ
#   df_size <- as.data.frame(table(Cluster = grpname))
#   p_size <- ggplot(df_size, aes(x = Cluster, y = Freq, fill = Cluster)) +
#     geom_col(width = 0.7, alpha = 0.9, show.legend = FALSE) +
#     geom_text(aes(label = Freq), vjust = -0.3, size = 3.6) +
#     labs(title = "Cluster sizes",
#          subtitle = paste0("Assignment by NbClust best k  |  ", suffix_tag),
#          x = "Cluster", y = "Count") +
#     theme_minimal(base_size = 12) +
#     theme(panel.grid.minor = element_blank(),
#           axis.title = element_text(face = "bold"),
#           plot.title = element_text(face = "bold"))
#   ggsave(file.path(out_dir, paste0("Cluster_sizes_", cindex, "_", suffix_tag, "_", ts_tag, ".png")),
#          p_size, width = 6.5, height = 5, dpi = 300)
#   ggsave(file.path(out_dir, paste0("Cluster_sizes_", cindex, "_", suffix_tag, "_", ts_tag, ".pdf")),
#          p_size, width = 6.5, height = 5, device = cairo_pdf)

#   list(best_nc = best_nc, grp = grpname, clustEst = clustEst)
# }

# # =========================
# # メイン：OH / FP を順に実行（旧3.3構成）
# # =========================
# for (ifname in dataset_files) {
#   # データセット名（OH / FP）推定：旧3.3の簡易推定を踏襲
#   dataset_stem <- sub("\\.csv$", "", ifname)
#   suffix_tag <- if (grepl("OH", ifname, ignore.case = TRUE)) "OH" else
#                 if (grepl("FP", ifname, ignore.case = TRUE)) "FP" else dataset_stem

#   # 出力フォルダ（データごと）— 旧3.3では1階層
#   out_dir <- file.path(root_out, paste0(suffix_tag))
#   dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

#   # 本改修でのみ追加：ログ保存用のサブフォルダ
#   d_logs <- file.path(out_dir, "99_logs")
#   dir.create(d_logs, showWarnings = FALSE, recursive = TRUE)

#   ts_tag <- format(Sys.time(), "%Y%m%d_%H%M%S")  # ファイル名用の時刻タグ

#   message("\n==============================")
#   message(sprintf("=== Processing: %s (%s) ===", ifname, suffix_tag))
#   message("==============================")

#   # 旧3.3：読み込み → 今回改修：3.2互換の前処理つき読み込み関数に置換
#   in_path <- file.path(base_dir, ifname)
#   readData <- read_numeric_matrix_3_2style(in_path)

#   # 前処理レポート出力（今回改修のみの追加）
#   removed_log <- attr(readData, "removed_log")
#   if (!is.null(removed_log) && nrow(removed_log) > 0) {
#     data.table::fwrite(
#       removed_log,
#       file.path(d_logs, sprintf("preprocess_report_%s_%s.csv", suffix_tag, ts_tag))
#     )
#   } else {
#     data.table::fwrite(
#       data.table(column=character(), reason=character()),
#       file.path(d_logs, sprintf("preprocess_report_%s_%s.csv", suffix_tag, ts_tag))
#     )
#   }

#   # 併せて runlog も保存（今回改修のみの追加）
#   runlog_path <- file.path(d_logs, sprintf("STEP3.3_runlog_%s_%s.txt", suffix_tag, ts_tag))
#   con <- file(runlog_path, open="wt"); sink(con, type="output"); on.exit({sink(); close(con)}, add=TRUE)
#   cat(sprintf("[RUN] %s %s\n", suffix_tag, ts_tag))
#   if (!is.null(removed_log)) {
#     cat("[PREPROCESS] removed summary:\n"); print(removed_log[, .N, by=reason])
#   }

#   # ===== MDS =====
#   m <- compute_mds(readData, k_max = 300)
#   mdsdata        <- m$mdsdata
#   eig_all        <- m$eig_all
#   share_all      <- m$share_all
#   share_pos_full <- m$share_pos_full
#   cum_all        <- m$cum_all
#   cum_pos        <- m$cum_pos

#   # 第1〜3軸の簡易レポート（全固有値基準）
#   p1_all <- share_all[1]; p2_all <- share_all[2]
#   cum3_all <- sum(share_all[1:min(3, length(share_all))], na.rm = TRUE)
#   cat("【全固有値基準】第1:", pct(p1_all), "  第2:", pct(p2_all),
#       "  1–2合計:", pct(p1_all + p2_all), "  1–3累積:", pct(cum3_all), "\n")

#   # 寄与一覧 CSV（全基準／正の基準）
#   df_eig_out <- data.frame(
#     Dim           = seq_along(eig_all),
#     Eigenvalue    = eig_all,
#     Share_all     = share_all,
#     Share_pos     = share_pos_full,
#     CumShare_all  = cum_all,
#     CumShare_pos  = cum_pos
#   )
#   write.csv(df_eig_out, file.path(out_dir, paste0("MDS_eigen_contributions_", suffix_tag, "_", ts_tag, ".csv")),
#             row.names = FALSE)

#   # 図（スクリープロット、Top50）— 旧3.3のまま
#   save_eigen_csv_and_plots(out_dir, suffix_tag, ts_tag, eig_all)

#   # ====== NbClust：指標ごと（全次元のmdsdata$pointsを使う可視化）======
#   for (cindex in index_list) {
#     cat("\n=== Index:", cindex, "===\n")
#     res <- run_nbclust_one_index(mds_points = mdsdata$points,
#                                  cindex = cindex,
#                                  out_dir = out_dir,
#                                  suffix_tag = suffix_tag,
#                                  ts_tag = ts_tag)
#     if (!is.null(res)) {
#       cat("Best.nc =", res$best_nc, "\n")
#       cat("Saved figures & assignment under: ", normalizePath(out_dir), "\n")
#     }
#   }

#   # ====== top3 vs cumeig（≥0.8）での一致度（ARI） ======
#   dims <- choose_dims(share_pos_full, cum_pos, target = 0.80)
#   k_top3   <- dims$top3
#   k_cumeig <- dims$cumeig

#   # 利用可能次元の上限でカット
#   avail_dims <- ncol(mdsdata$points)
#   k_top3   <- min(k_top3,   avail_dims)
#   k_cumeig <- min(k_cumeig, avail_dims)

#   coords_list <- list(
#     top3   = mdsdata$points[, 1:k_top3,   drop = FALSE],
#     cumeig = mdsdata$points[, 1:k_cumeig, drop = FALSE]
#   )

#   # 座標CSV（条件ごと）
#   for (cond in names(coords_list)) {
#     write.csv(coords_list[[cond]],
#               file.path(out_dir, paste0("MDScoords_", cond, "_", suffix_tag, "_", ts_tag, ".csv")))
#   }

#   # 条件×指標でクラスタリングして保存（旧3.3の命名を踏襲）
#   assignments <- list()
#   for (cond in names(coords_list)) {
#     X <- coords_list[[cond]]
#     rownames(X) <- rownames(mdsdata$points)

#     for (cindex in index_list) {
#       clustEst <- tryCatch({
#         NbClust(data = X, diss = NULL, distance = "euclidean",
#                 min.nc = 2, max.nc = min(100, nrow(X)-1), method = "ward.D2", index = cindex)
#       }, error = function(e) { warning(paste("failed:", cond, cindex, e$message)); NULL })
#       if (is.null(clustEst)) next

#       grp <- as.factor(clustEst$Best.partition)
#       # 割当て保存
#       fn_assign2 <- paste0("ClusterAssign_", cond, "_", cindex, "_",
#                            suffix_tag, "_", ts_tag, ".csv")
#       write.csv(data.frame(Variable = names(grp), Cluster = grp),
#                 file.path(out_dir, fn_assign2), row.names = FALSE)

#       assignments[[paste(cond, cindex, sep = "_")]] <- grp
#     }
#   }

#   # ARI: top3 と cumeig の一致度
#   ari_rows <- lapply(index_list, function(ix) {
#     a <- assignments[[paste0("top3_", ix)]]
#     b <- assignments[[paste0("cumeig_", ix)]]
#     ari <- if (!is.null(a) && !is.null(b)) mclust::adjustedRandIndex(a, b) else NA_real_
#     data.frame(Index = ix, ARI_top3_vs_cumeig = ari)
#   })
#   df_ari <- do.call(rbind, ari_rows)
#   write.csv(df_ari, file.path(out_dir, paste0("ARI_top3_vs_cumeig_", suffix_tag, "_", ts_tag, ".csv")),
#             row.names = FALSE)
#   print(df_ari)

#   cat("\n✅ Done: ", suffix_tag, " → ", normalizePath(out_dir), "\n")
# }

# cat("\n🎉 全データの処理が完了しました → ", normalizePath(root_out), "\n")




=== Processing: preprocessed_features_OH.csv (OH) ===


[Info] rows=1046, cols=394

[Info] Non-NA percentage: 98.69%

Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 3 rows containing missing values or values outside the scale range
(`geom_path()`)."
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 3 rows containing missing values or values outside the scale range
(`geom_path()`)."


=== Processing: preprocessed_features_FP.csv (FP) ===


[Info] rows=1046, cols=251

[Info] Non-NA percentage: 97.95%

Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an 

Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 83 rows containing missing values or values outside the scale range
(`geom_path()`)."
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an

Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 83 rows containing missing values or values outside the scale range
(`geom_path()`)."
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an

Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 85 rows containing missing values or values outside the scale range
(`geom_path()`)."


ERROR: Error in close.connection(con): invalid connection
